# Water Body Segmentation using U-Net
## Binary Segmentation: Water vs Non-Water

## 1. Setup - Environment & Libraries

In [ ]:
# Check GPU availability
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Install segmentation-models-pytorch
!pip install segmentation-models-pytorch -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/water_segmentation', exist_ok=True)
print("Drive mounted. Results will be saved to: /content/drive/MyDrive/water_segmentation/")

In [ ]:
# Setup Kaggle API
!pip install kaggle -q

from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print("Kaggle API configured!")

In [ ]:
# Download dataset
!kaggle datasets download -d franciscoescobar/satellite-images-of-water-bodies -p /content/

# Unzip
!unzip -q /content/satellite-images-of-water-bodies.zip -d /content/water_dataset
print("Dataset downloaded and extracted!")

In [ ]:
# Import all libraries
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import random
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.functional as TF
import segmentation_models_pytorch as smp

# Set paths
IMAGE_DIR = '/content/water_dataset/Water Bodies Dataset/Images'
MASK_DIR = '/content/water_dataset/Water Bodies Dataset/Masks'
SAVE_DIR = '/content/drive/MyDrive/water_segmentation'

print(f"Images found: {len(os.listdir(IMAGE_DIR))}")
print(f"Masks found: {len(os.listdir(MASK_DIR))}")